In [55]:
import pandas as pd
import numpy as np
from IPython.display import display
# Hospital Consumer Assessment of Healthcare Providers and Systems
pt_surveys = pd.read_csv('../data/raw/HCAHPS_patient_surveys.csv')
cl_general_info = pd.read_csv("../data/processed/general_info_clean.csv")
df = pt_surveys.copy()

df['Patient Survey Star Rating'] = df['Patient Survey Star Rating'].replace('Not Available', np.nan)
df['Patient Survey Star Rating'] = df['Patient Survey Star Rating'].replace('Not Applicable', np.nan)
df['Patient Survey Star Rating'] = pd.to_numeric(df['Patient Survey Star Rating'], errors='coerce')

# Define which HCAHPS Measure IDs map to which category
measure_map = {
    "H_CLEAN_STAR_RATING": ("Cleanliness", "Star Rating"),
    "H_CLEAN_LINEAR_SCORE": ("Cleanliness", "Linear Mean"),
    "H_COMP_1_STAR_RATING": ("Nurse Communication", "Star Rating"),
    "H_COMP_1_LINEAR_SCORE": ("Nurse Communication", "Linear Mean"),
    "H_COMP_2_STAR_RATING": ("Doctor Communication", "Star Rating"),
    "H_COMP_2_LINEAR_SCORE": ("Doctor Communication", "Linear Mean"),
    "H_COMP_3_STAR_RATING": ("Responsiveness of Hospital Staff", "Star Rating"),
    "H_COMP_3_LINEAR_SCORE": ("Responsiveness of Hospital Staff", "Linear Mean"),
    "H_COMP_4_STAR_RATING": ("Pain Management", "Star Rating"),
    "H_COMP_4_LINEAR_SCORE": ("Pain Management", "Linear Mean"),
    "H_COMP_5_STAR_RATING": ("Communication about Medicines", "Star Rating"),
    "H_COMP_5_LINEAR_SCORE": ("Communication about Medicines", "Linear Mean"),
    "H_COMP_6_STAR_RATING": ("Discharge Information", "Star Rating"),
    "H_COMP_6_LINEAR_SCORE": ("Discharge Information", "Linear Mean"),
    "H_QUIET_STAR_RATING": ("Quietness", "Star Rating"),
    "H_QUIET_LINEAR_SCORE": ("Quietness", "Linear Mean"),
    "H_HSP_RATING_STAR_RATING": ("Overall Hospital Rating", "Star Rating"),
    "H_HSP_RATING_LINEAR_SCORE": ("Overall Hospital Rating", "Linear Mean"),
    "H_RECMND_STAR_RATING": ("Recommend Hospital", "Star Rating"),
    "H_RECMND_LINEAR_SCORE": ("Recommend Hospital", "Linear Mean"),
}

# Keep only the rows for measures we care about
df_filtered = df[df["HCAHPS Measure ID"].isin(measure_map.keys())].copy()

# Clean up the numeric columns before pivoting
for col in ["Patient Survey Star Rating", "HCAHPS Linear Mean Value"]:
    df_filtered[col] = df_filtered[col].replace(
        ["Not Available", "Not Applicable"], np.nan
    )
    df_filtered[col] = pd.to_numeric(df_filtered[col], errors="coerce")

# Pivot Star Ratings
star = df_filtered[
    df_filtered["HCAHPS Measure ID"].str.contains("STAR_RATING")
].pivot_table(
    index=["Facility ID", "Facility Name"],
    columns="HCAHPS Measure ID",
    values="Patient Survey Star Rating",
)

# Pivot Linear Means
linear = df_filtered[
    df_filtered["HCAHPS Measure ID"].str.contains("LINEAR")
].pivot_table(
    index=["Facility ID", "Facility Name"],
    columns="HCAHPS Measure ID",
    values="HCAHPS Linear Mean Value",
)

# Combine and reset index
result = pd.concat([star, linear], axis=1).reset_index()


# Rename columns using measure_map, with fallback for Facility ID/Name
def make_multiindex(cols):
    new = []
    for c in cols:
        if c == "Facility Name":
            new.append(("Facility Name", ""))
        elif c == "Facility ID":
            new.append(("Facility ID", ""))
        else:
            new.append(measure_map.get(c, (c, "")))
    return pd.MultiIndex.from_tuples(new)


result.columns = make_multiindex(result.columns)

# Reorder so Facility Name comes first, Facility ID second
result = result[
    [("Facility Name", ""), ("Facility ID", "")]
    + [
        c
        for c in result.columns
        if c not in [("Facility Name", ""), ("Facility ID", "")]
    ]
]

display(result)

print(df.info())
display(df.head(10))
display(cl_general_info.iloc[:,32:34])

print(cl_general_info.columns.to_list())
display(cl_general_info.info())

C:\Users\swyne\AppData\Local\Temp\ipykernel_36580\1698502900.py:5: DtypeWarning: Columns (12,14,17,19) have mixed types. Specify dtype option on import or set low_memory=False.
  pt_surveys = pd.read_csv('../data/raw/HCAHPS_patient_surveys.csv')


,Facility Name,Facility ID,Cleanliness,Nurse Communication,Doctor Communication,Communication about Medicines,Discharge Information,Overall Hospital Rating,Quietness,Recommend Hospital,Cleanliness,Nurse Communication,Doctor Communication,Communication about Medicines,Discharge Information,Overall Hospital Rating,Quietness,Recommend Hospital
,,,Star Rating,Star Rating,Star Rating,Star Rating,Star Rating,Star Rating,Star Rating,Star Rating,Linear Mean,Linear Mean,Linear Mean,Linear Mean,Linear Mean,Linear Mean,Linear Mean,Linear Mean
0,BIRMINGHAM VA MEDICAL CENTER,01014F,3.0,4.0,4.0,4.0,3.0,4.0,3.0,4.0,86.0,92.0,93.0,83.0,86.0,90.0,82.0,88.0
1,VA CENTRAL ALABAMA HEALTHCARE SYSTEM - MONTGOMERY,01019F,3.0,2.0,3.0,2.0,2.0,3.0,3.0,2.0,86.0,88.0,91.0,71.0,81.0,88.0,82.0,83.0
2,673rd Medical Group (Joint Base Elmendorf-Rich...,02013F,2.0,5.0,4.0,4.0,4.0,4.0,4.0,5.0,82.0,95.0,94.0,83.0,88.0,92.0,84.0,92.0
3,PHOENIX VA MEDICAL CENTER,03012F,3.0,3.0,3.0,3.0,4.0,3.0,2.0,3.0,84.0,91.0,91.0,78.0,88.0,86.0,74.0,85.0
4,VA S. ARIZONA HEALTHCARE SYSTEM,03013F,4.0,4.0,3.0,4.0,3.0,4.0,2.0,5.0,88.0,92.0,91.0,81.0,86.0,91.0,75.0,92.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3178,GEORGE WASHINGTON UNIV HOSPITAL,90001,2.0,1.0,3.0,1.0,1.0,2.0,2.0,2.0,80.0,85.0,90.0,68.0,78.0,84.0,73.0,83.0
3179,HOWARD UNIVERSITY HOSPITAL CORP,90003,2.0,1.0,2.0,2.0,2.0,2.0,3.0,1.0,82.0,84.0,88.0,72.0,80.0,82.0,82.0,76.0
3180,MEDSTAR GEORGETOWN UNIVERSITY HOSPITAL,90004,3.0,3.0,3.0,2.0,3.0,3.0,3.0,4.0,84.0,90.0,91.0,74.0,85.0,87.0,83.0,87.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 325652 entries, 0 to 325651
Data columns (total 22 columns):
 #   Column                                 Non-Null Count   Dtype  
---  ------                                 --------------   -----  
 0   Facility ID                            325652 non-null  object 
 1   Facility Name                          325652 non-null  object 
 2   Address                                325652 non-null  object 
 3   City/Town                              325652 non-null  object 
 4   State                                  325652 non-null  object 
 5   ZIP Code                               325652 non-null  int64  
 6   County/Parish                          325652 non-null  object 
 7   Telephone Number                       325652 non-null  object 
 8   HCAHPS Measure ID                      325652 non-null  object 
 9   HCAHPS Question                        325652 non-null  object 
 10  HCAHPS Answer Description              325652 non-null  

,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,HCAHPS Measure ID,HCAHPS Question,...,Patient Survey Star Rating Footnote,HCAHPS Answer Percent,HCAHPS Answer Percent Footnote,HCAHPS Linear Mean Value,Number of Completed Surveys,Number of Completed Surveys Footnote,Survey Response Rate Percent,Survey Response Rate Percent Footnote,Start Date,End Date
0,10001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,H_COMP_1_A_P,"Patients who reported that their nurses ""Alway...",...,NaN,75,NaN,Not Applicable,596,NaN,16,NaN,4/1/2024,3/31/2025
1,10001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,H_COMP_1_SN_P,"Patients who reported that their nurses ""Somet...",...,NaN,6,NaN,Not Applicable,596,NaN,16,NaN,4/1/2024,3/31/2025
2,10001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,H_COMP_1_U_P,"Patients who reported that their nurses ""Usual...",...,NaN,19,NaN,Not Applicable,596,NaN,16,NaN,4/1/2024,3/31/2025
3,10001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,H_COMP_1_LINEAR_SCORE,Nurse communication - linear mean score,...,NaN,Not Applicable,NaN,90,596,NaN,16,NaN,4/1/2024,3/31/2025
4,10001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,H_COMP_1_STAR_RATING,Nurse communication - star rating,...,NaN,Not Applicable,NaN,Not Applicable,596,NaN,16,NaN,4/1/2024,3/31/2025
5,10001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,H_NURSE_RESPECT_A_P,"Patients who reported that their nurses ""Alway...",...,NaN,84,NaN,Not Applicable,596,NaN,16,NaN,4/1/2024,3/31/2025
6,10001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,H_NURSE_RESPECT_SN_P,"Patients who reported that their nurses ""Somet...",...,NaN,3,NaN,Not Applicable,596,NaN,16,NaN,4/1/2024,3/31/2025
7,10001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,H_NURSE_RESPECT_U_P,"Patients who reported that their nurses ""Usual...",...,NaN,13,NaN,Not Applicable,596,NaN,16,NaN,4/1/2024,3/31/2025
8,10001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,H_NURSE_LISTEN_A_P,"Patients who reported that their nurses ""Alway...",...,NaN,71,NaN,Not Applicable,596,NaN,16,NaN,4/1/2024,3/31/2025
9,10001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,H_NURSE_LISTEN_SN_P,"Patients who reported that their nurses ""Somet...",...,NaN,8,NaN,Not Applicable,596,NaN,16,NaN,4/1/2024,3/31/2025


,Pt Exp Group Measure Count,Count of Facility Pt Exp Measures
0,8,8.0
1,8,8.0
2,8,8.0
3,8,8.0
4,8,8.0
...,...,...
2861,8,8.0
2862,8,8.0
2863,8,NaN
2864,8,8.0


['Facility ID', 'Facility Name', 'Address', 'City/Town', 'State', 'ZIP Code', 'County/Parish', 'Telephone Number', 'Hospital Type', 'Hospital Ownership', 'Emergency Services', 'Meets criteria for birthing friendly designation', 'Hospital overall rating', 'Hospital overall rating footnote', 'MORT Group Measure Count', 'Count of Facility MORT Measures', 'Count of MORT Measures Better', 'Count of MORT Measures No Different', 'Count of MORT Measures Worse', 'MORT Group Footnote', 'Safety Group Measure Count', 'Count of Facility Safety Measures', 'Count of Safety Measures Better', 'Count of Safety Measures No Different', 'Count of Safety Measures Worse', 'Safety Group Footnote', 'READM Group Measure Count', 'Count of Facility READM Measures', 'Count of READM Measures Better', 'Count of READM Measures No Different', 'Count of READM Measures Worse', 'READM Group Footnote', 'Pt Exp Group Measure Count', 'Count of Facility Pt Exp Measures', 'Pt Exp Group Footnote', 'TE Group Measure Count', 'Co

None